In [1]:
import openai
from qdrant_client import QdrantClient

### Embedding Function

In [2]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

### Retrieval Function

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [4]:
def retrieve_data(query,qdrant_client, k=5) :

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points : 
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)
    
    return {
        "retrieved_context_ids" : retrieved_context_ids,
        "retrieved_context" : retrieved_context,
        "retrieved_context_ratings" : retrieved_context_ratings,
        "similarity_scores" : similarity_scores
    }

In [5]:
retrieved_context = retrieve_data("what kind of earphones can i get ?", qdrant_client, k =10)

In [6]:
retrieved_context

{'retrieved_context_ids': ['B0BCVM5YCJ',
  'B08P3HHBTS',
  'B0080HVBHQ',
  'B097SZZZGW',
  'B0BYSJYXWW',
  'B00I0FXH20',
  'B00F5L6CAQ',
  'B0753KDNQ9',
  'B07DFBHDBT',
  'B07R71SY87'],
 'retrieved_context': ["iClever BTH02 Kids Headphones, Kids Wireless Headphones with MIC, 22H Playtime, Bluetooth 5.0 & Stereo Sound, Foldable, Adjustable Headband, Childrens Headphones for iPad Tablet Home School, Purple 【Stereo Sound & 94dB Volume Limiting】: Full-coverage padded earmuffs will wrap your little ones in powerful, high-quality sound. Your child's safety is our priority. iClever kids Bluetooth headphones max out the volume level at 94db for immersive yet safe listening. They'll get lost in the music, movie, game, and more! 【Bluetooth 5.0 & Built-In Microphone】: Say goodbye to tangled wired headphones. iClever kids wireless headphones adopts the latest Bluetooth 5.0 to maintain stable connection. Play, pause, answer, and end calls with a touch of a button. The built-in microphone of kids he

### Formate retrieved context function

In [10]:
def process_context(context):

    formatted_context = ""

    for id, chunk , rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context['retrieved_context_ratings']):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [11]:
preprocessed_context = process_context(retrieved_context)

In [12]:
print(preprocessed_context)

- ID: B0BCVM5YCJ, rating: 4.5, description: iClever BTH02 Kids Headphones, Kids Wireless Headphones with MIC, 22H Playtime, Bluetooth 5.0 & Stereo Sound, Foldable, Adjustable Headband, Childrens Headphones for iPad Tablet Home School, Purple 【Stereo Sound & 94dB Volume Limiting】: Full-coverage padded earmuffs will wrap your little ones in powerful, high-quality sound. Your child's safety is our priority. iClever kids Bluetooth headphones max out the volume level at 94db for immersive yet safe listening. They'll get lost in the music, movie, game, and more! 【Bluetooth 5.0 & Built-In Microphone】: Say goodbye to tangled wired headphones. iClever kids wireless headphones adopts the latest Bluetooth 5.0 to maintain stable connection. Play, pause, answer, and end calls with a touch of a button. The built-in microphone of kids headphones provides hands-free control and easy operation. 【Hours of Uninterrupted Listening】: Enjoy enough listening time with up to 22 hours of music, study, movies, 

In [17]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions :
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

In [18]:
prompt = build_prompt(preprocessed_context, "What kind of earphones can i get?")

In [19]:
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions :
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
- ID: B0BCVM5YCJ, rating: 4.5, description: iClever BTH02 Kids Headphones, Kids Wireless Headphones with MIC, 22H Playtime, Bluetooth 5.0 & Stereo Sound, Foldable, Adjustable Headband, Childrens Headphones for iPad Tablet Home School, Purple 【Stereo Sound & 94dB Volume Limiting】: Full-coverage padded earmuffs will wrap your little ones in powerful, high-quality sound. Your child's safety is our priority. iClever kids Bluetooth headphones max out the volume level at 94db for immersive yet safe listening. They'll get lost in the music, movie, game, and more! 【Bluetooth 5.0 & Built-In Microphone】: Say goodbye to tangled wired headphones. iClever kids wireless headphones adopts the latest Blue

### Generate answer function

In [24]:
def generate_answer(prompt) :

    response = openai.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[{"role":"system", "content": prompt}],
        reasoning_effort="low"
    )

    return response.choices[0].message.content

In [25]:
print(generate_answer(prompt))

You can get the **iClever BTH02 Kids Wireless Headphones** (ID: B0BCVM5YC). They offer:

- Bluetooth 5.0 and a built-in microphone  
- Up to 22 hours of playtime  
- Stereo sound with a 94 dB volume limit  
- Foldable, adjustable design with padded ear cushions  
- 3.5 mm wired backup connection  
- Purple color


### Combined RAG pipeline

In [26]:
def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer

In [28]:
print(rag_pipeline("What kind of earphones can I get with ratings above 4.0?"))

You can get the **iClever BTH02 Kids Wireless Headphones** with a **4.5 rating**. They feature:

- Bluetooth 5.0 and built-in microphone
- Up to 22 hours of playtime
- 94 dB volume limiting for safer listening
- Foldable, adjustable design
- 3.5 mm backup audio jack
- Purple color
